<a href="https://colab.research.google.com/github/allengr220/cognitive-ledger/blob/main/longevity/benchmark_levine_no_crp_cohort_builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Benchmark Levine-No-CRP Cohort Builder — Top Spec

## Purpose

This notebook creates the **aligned benchmark cohort** used for comparing:

- **PhysioRisk_2axis**
- **TotalRisk_2axis**
- **PhenoAge (CRP-excluded proxy)**
- **PhenoAgeAccel**

The goal is to define a single, internally consistent cohort that supports both the locked PhysioRisk pipeline and the Levine-style comparator under the same survival framework.

---

## Role in architecture

This notebook is a **cohort-definition step**, not a modeling notebook.

It does **not**:
- fit BaselineRisk
- residualize biomarkers
- fit PhysioRisk
- compute final benchmark metrics

It only produces the cohort artifact that downstream notebooks will consume.

---

## Input dataset

Source:
- `nhanes_1999_2016_extended_mortality_model_dataset.parquet`

This master dataset contains:
- mortality linkage
- cycle metadata
- core demographic variables
- extended biomarker variables

---

## Output artifact

This notebook outputs:

- `benchmark_levine_no_crp_cohort.parquet`

This file is the canonical cohort input for downstream benchmark processing.

---

## Cohort definition

Rows retained must satisfy all of the following:

### Survival eligibility
- `PERMTH_INT` is non-missing
- `MORTSTAT` is non-missing

### Required demographic / survival variables
- `SEQN`
- `RIDAGEYR`
- `RIAGENDR`
- `PERMTH_INT`
- `MORTSTAT`
- `cycle_start_year`

### Required biomarker variables
- `LBXSAL`
- `LBXSCR`
- `LBXGLU`
- `LBXLYPCT`
- `LBXMCV`
- `LBXRDW`
- `LBXSAPSI`
- `LBXWBCSI`

These variables support:
- the 2-axis PhysioRisk model
- a Levine-style benchmark without CRP

---

## Explicit exclusion of CRP

CRP is **not included** in this primary benchmark cohort.

Reason:
- NHANES 1999–2016 does not provide a stable CRP field across all cycles
- 1999–2010 uses `LBXCRP` (standard CRP, mg/dL)
- 2011–2014 has no CRP field
- 2015–2016 uses `LBXHSCRP` (hsCRP, mg/L)

Because this introduces:
- variable identity shift
- unit shift
- missing mid-period block

CRP is excluded here to preserve a clean, cycle-consistent benchmark cohort.

A separate CRP-enabled sensitivity benchmark may be built later on the 1999–2010 subset.

---

## Expected properties of the output cohort

Current benchmark cohort:
- **N = 19,829**
- **Deaths = 2,936**

Cycle coverage:
- 1999
- 2001
- 2003
- 2005
- 2007
- 2009
- 2011
- 2013
- 2015

This preserves full 1999–2016 coverage while maintaining consistent biomarker availability.

---

## Downstream usage

This cohort artifact will be used in the following sequence:

1. **Split-aware baseline builder**
   - fit BaselineRisk on train only
   - apply unchanged to train and test
   - export baseline-scored train/test benchmark parquet files

2. **Canonical 2-axis PhysioRisk notebook**
   - read baseline-scored benchmark train/test files
   - residualize biomarkers
   - build PhysioRisk_2axis
   - export scored benchmark train/test files

3. **Aligned benchmark comparison notebook**
   - compute PhenoAge (CRP-excluded proxy)
   - compute PhenoAgeAccel (train-only age regression)
   - compare against:
     - BaselineRisk
     - PhysioRisk_2axis
     - TotalRisk_2axis

---

## Invariants

1. This notebook defines the benchmark cohort only
2. No model fitting occurs here
3. No baseline estimation occurs here
4. No residualization occurs here
5. No CRP harmonization occurs here
6. The output cohort must remain fixed for all downstream aligned benchmark analyses

---

## Scientific rationale

This benchmark cohort is designed to answer a specific question:

> How does the locked 2-axis PhysioRisk model compare to a Levine-style biological age comparator when both are evaluated on the same biomarker-consistent, mortality-linked cohort?

By excluding CRP from the primary benchmark, this cohort prioritizes:
- internal consistency
- fair comparison
- reviewer defensibility

rather than maximal fidelity to the original Levine specification.

---

## Status

This cohort is the **primary aligned no-CRP benchmark cohort** for the main PhysioRisk vs PhenoAge / PhenoAgeAccel comparison.

In [ ]:
import pandas as pd

MASTER_PATH = "/content/drive/MyDrive/LRE/data/pub/nhanes_1999_2016_extended_mortality_model_dataset.parquet"
OUT_PATH = "/content/drive/MyDrive/LRE/data/pub/benchmark_levine_no_crp_cohort.parquet"

df = pd.read_parquet(MASTER_PATH).copy()

# survival rows only (defensive; should already be true here)
df = df[df["PERMTH_INT"].notna() & df["MORTSTAT"].notna()].copy()

COLS = [
    "SEQN","PERMTH_INT","MORTSTAT","RIDAGEYR","RIAGENDR","cycle_start_year",
    "LBXSAL","LBXSCR","LBXGLU","LBXLYPCT","LBXMCV","LBXRDW","LBXSAPSI","LBXWBCSI",
]

benchmark_df = df.dropna(subset=COLS)[COLS].drop_duplicates("SEQN").copy()

benchmark_df.to_parquet(OUT_PATH, index=False)

print("Shape:", benchmark_df.shape)
print("Deaths:", int(benchmark_df["MORTSTAT"].sum()))
print("Cycles:")
print(benchmark_df["cycle_start_year"].value_counts().sort_index())

benchmark_df.to_parquet(OUT_PATH, index=False)
print("Saved:", OUT_PATH)